# MPI–ORAS validation
Comparison between MPI climate model and ORAS5

## Imports

In [ ]:
import warnings
import datetime
import matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl
import cartopy.crs as ccrs
import numpy as np
import seaborn as sns
import xarray as xr
import tqdm
import pathlib
import cmocean
import pandas as pd
import os
import scipy.stats
import calendar
import copy

# import cartopy.util

# Import custom modules
from src.XRO import XRO, xcorr
import src.XRO_utils
import src.utils

## set plotting specs
sns.set(rc={"axes.facecolor": "white", "axes.grid": False})

## bump up DPI
mpl.rcParams["figure.dpi"] = 100

## get filepaths
DATA_FP = pathlib.Path(os.environ["DATA_FP"])
SAVE_FP = pathlib.Path(os.environ["SAVE_FP"])

## funcs

In [ ]:
def plot_spectrum(sigma, omega):
    """plot spectrum with guidelines"""

    ## specify how many to plot
    n = 60
    s = 15

    ## axis limits
    ylim = [-0.5, 0.5]
    xlim = [-2.5, 0.1]
    yticks = [-0.25, 0, 0.25]

    ## params for guidelines
    lw_ = 1.5
    ls_ = "-"
    alpha = 0.4

    ## plot gamma
    fig, ax = plt.subplots(figsize=(4, 3))

    ## set axis limits
    ax.set_ylim(ylim)
    ax.set_xlim(xlim)
    ax.set_yticks(yticks)
    ax.set_xticks([-2, -1, -0.5], labels=["1/2", "1", "2"])
    ax.set_xlabel(f"$e$-fold timescale (yrs)")

    ## plot data
    ax.scatter(sigma[:n], omega[:n], s=s, c="k")

    ## add guide lines
    add_guide = lambda ax, f, c, l: ax.axhline(
        f, c=c, lw=lw_, ls=ls_, alpha=alpha, label=l
    )
    ax.axvline(0, c="k", lw=lw_)

    ## plot annual cycle harmonics
    for i, freq in enumerate(np.arange(-2, 3)):
        label = "Annual" if not i else None
        add_guide(ax, freq, c="k", l=label)

    ## plot ENSO modes
    for i, freq in enumerate([-1 / 4, 1 / 4]):
        label = "ENSO" if not i else None
        add_guide(ax, freq, c="r", l=label)

    # axs[1].set_yticks([])
    ax.legend()

    return fig, ax

## Load data

In [ ]:
## load SST data and truncate to 20 modes
forced, anom = src.utils.load_consolidated()
anom = anom[["sst", "sst_comp"]].isel(mode=slice(None, 30))

## window it
anom = src.utils.get_windowed(anom, window_size=480, stride=120)
anom = anom.isel(year=slice(None, -1))

## LIM

In [ ]:
## get X and Y data
prep = lambda x: x["sst"].isel(year=10).stack(sample=["member", "time"])
X = prep(anom.isel(time=slice(None, -1)))
Y = prep(anom.isel(time=slice(1, None)))

## get month labels
month_idx = X.time.dt.month.values - 1

X = X.values
Y = Y.values

### Demo LIM fitting:

In [ ]:
import src.lim

## fit LIM
lim = src.lim.LIM_CS(X=X, Y=Y, month_labels=month_idx)

## get spectrum
sigma = 12 * lim.sigma
omega = 12 / (2 * np.pi) * lim.omega

In [ ]:
fig, ax = plot_spectrum(sigma=sigma, omega=omega)
ax.set_ylim([-0.4, 0.4])
ax.set_xlim([-1.5, 0.1])
ax.set_ylabel(r"$\omega$ (year$^{-1}$)")
plt.show()

In [ ]:
np.allclose(lim.U[0].T @ lim.V[0], np.eye(30))

## Fit over time

In [ ]:
## helper func
stack = lambda x: x.stack(sample=["member", "time"])

## coordinates for data
coords = dict(mode=anom.mode.values)
coords_pat = dict(
    month=np.arange(1, 13), mode=anom.mode.values, n_mode=anom.mode.values
)

## empty lists to hold results
patterns = []
sigmas = []
omegas = []

## loop thru years
for yr in tqdm.tqdm(anom.year):

    ## get data
    XY = anom["sst"].sel(year=yr)

    ## separate X, Y
    X = stack(XY.isel(time=slice(None, -1)))
    Y = stack(XY.isel(time=slice(1, None)))
    month_idx = X.time.dt.month.values - 1

    ## fit LIM
    lim = src.lim.LIM_CS(X=X.values, Y=Y.values, month_labels=month_idx)

    ## get sigma/omega
    sigma = 12 * lim.sigma
    omega = 12 / (2 * np.pi) * lim.omega

    ## get spectrum
    sigmas.append(xr.DataArray(sigma, coords))
    omegas.append(xr.DataArray(omega, coords))
    patterns.append(xr.DataArray(lim.U, coords_pat, dims=list(coords_pat)))
    # omega = 12 / (2 * np.pi) * lim.omega

## put in dataarrays
concat = lambda x: xr.concat(x, dim=anom.year)
res = xr.merge(
    [
        concat(sigmas).rename("sigma"),
        concat(omegas).rename("omega"),
        concat(patterns).rename("pattern"),
    ]
)

### Get spatial structure

In [ ]:
## get patterns
patterns = xr.merge(
    [
        res["pattern"],
        anom["sst_comp"].rename("pattern_comp"),
    ]
)

## trim in mode space
patterns = patterns.isel(n_mode=slice(None, 10))

## reconstruct
res["patterns"] = src.utils.reconstruct_wrapper(patterns)["pattern"]

## trim
res = res.sel(n_mode=patterns.n_mode)

In [ ]:
def plot_sst2(ax, data, dx=0.2, nlev=5):
    """plot data on ax"""

    cp = ax.contourf(
        data.longitude,
        data.latitude,
        data,
        cmap="cmo.balance",
        levels=make_cb_range(dx, nlev),
        transform=ccrs.PlateCarree(),
        extend="both",
    )

    return cp


import cartopy.crs as ccrs


def make_cb_range(dx, nlevs):
    amp = dx * nlevs - dx / 2

    lev1 = np.arange(0, amp + dx / 2, dx) + dx / 2
    lev0 = -lev1[::-1]
    return np.concatenate([lev0, lev1])

#### ENSO mode

In [ ]:
idx = dict(n_mode=0, month=2)

## rotation angle
theta = 7 * np.pi / 10

fig = plt.figure(figsize=(6, 4.5), layout="constrained")
format_func = lambda ax,: src.utils.plot_setup_pac(ax, max_lat=21, lon_range=(140, 285))
axs = src.utils.subplots_with_proj(fig, nrows=3, ncols=1, format_func=format_func)

cp00 = plot_sst2(
    axs[0, 0],
    (np.exp(1j * theta) * res["patterns"].isel(idx).isel(year=0)).real,
    dx=0.1,
)

cp10 = plot_sst2(
    axs[1, 0],
    (-np.exp(1j * theta) * res["patterns"].isel(idx).isel(year=10)).real,
    dx=0.1,
)

cp20 = plot_sst2(
    axs[2, 0],
    (np.exp(1j * theta) * res["patterns"].isel(idx).isel(year=-1)).real,
    dx=0.1,
)

ax_kwargs = dict(ls="--", c="k", lw=1)
for ax in axs.flatten():
    for t in [-5, 5]:
        ax.axhline(t, **ax_kwargs)

# cp10 = plot_sst2(axs[1, 0], enso_pattern0, dx=dxs[1])

#### Secondary mode

In [ ]:
idx = dict(n_mode=2, month=0)

## rotation angle
theta = -1 * np.pi / 10

fig = plt.figure(figsize=(6, 4.5), layout="constrained")
format_func = lambda ax,: src.utils.plot_setup_pac(ax, max_lat=21, lon_range=(140, 285))
axs = src.utils.subplots_with_proj(fig, nrows=3, ncols=1, format_func=format_func)

cp00 = plot_sst2(axs[0, 0], res["patterns"].isel(idx).isel(year=0).real, dx=0.1)

cp10 = plot_sst2(axs[1, 0], -res["patterns"].isel(idx).isel(year=10).real, dx=0.1)

cp20 = plot_sst2(axs[2, 0], -res["patterns"].isel(idx).isel(year=-1).real, dx=0.1)

# cp10 = plot_sst2(axs[1, 0], enso_pattern0, dx=dxs[1])

### time evolution

In [ ]:
## get fractional change
get_delta = lambda x: x / x.isel(year=0) - 1

fig, axs = plt.subplots(1, 2, figsize=(6, 3), layout="constrained")

axs[0].plot(res.year, get_delta(res.sigma.isel(mode=0)))
axs[1].plot(res.year, get_delta(res.omega.isel(mode=0)))

for ax in axs:
    ax.axhline(0, ls="--", c="k", lw=1)

src.utils.set_lims(axs)
src.utils.add_vticks(axs, xticks=[1870, 2010, 2080], xlines=[2010])
axs[1].set_yticks([])
axs[0].set_title("Damping rate")
axs[1].set_title("Freq")
axs[0].set_ylabel(r"Frac. change")

plt.show()

## analysis

### spatial structure over time

### plot projection on El Niño/La Niña

### Sigma over time

## Old

Reconstruction functions

In [ ]:
def reconstruct(scores, other_coord=None):
    """reconstruct projected data"""

    ## get reconstructions
    kwargs = dict(other_coord=other_coord)
    T_recon = reconstruct_helper(scores[:nmodes], model=eofs_sst, **kwargs)
    h_recon = reconstruct_helper(scores[nmodes:], model=eofs_ssh, **kwargs)

    return xr.merge([T_recon.rename("T"), h_recon.rename("h")]).real


def reconstruct_helper(scores, model, other_coord=None):
    """reconstruct scores for given model"""

    ## put raw scores into xarray
    n = scores.shape[1]
    scores_xr = xr.zeros_like(
        model.scores().isel(member=0, time=slice(None, n))
    ).transpose("mode", ...)
    scores_xr.values[:nmodes] = scores

    ## do inverse transform
    data = model.inverse_transform(scores_xr)

    ## rename coord
    if other_coord is None:
        other_coord = pd.Index(np.arange(n), name="n")
    data = data.rename({"time": other_coord.name})
    data[other_coord.name] = other_coord.values

    return data

Find peak angle for El Niño

In [ ]:
## specify month to find peak index
peak_month_idx = 11

## get test pts
theta_test = np.arange(0, 2 * np.pi, np.pi / 32)
varphi_test = varphi.std() * np.exp(1j * theta_test)
VtX = np.stack([varphi_test, np.conj(varphi_test)], axis=0)

## Get recon (EOF space)
recon = Uk[peak_month_idx] @ VtX

## Get recon (real space)
recon_xr = reconstruct(recon.real, other_coord=pd.Index(theta_test, name="theta"))
nino_recon = src.utils.get_nino3(recon_xr["T"])

## get theta for maximimum Niño in peak month
theta_max = recon_xr.theta.isel(theta=nino_recon.argmax("theta")).values.item()

Get "real" reconstruction

In [ ]:
## get difference from max
lags_months = np.arange(-12, 13)
lags_years = lags_months / 12
delta_theta = 2 * np.pi * omega[enso_idx] * lags_years

## get data
month_idx_test = np.mod(peak_month_idx + lags_months, 12)
theta_test = np.mod(theta_max + delta_theta, 2 * np.pi)
varphi_test = varphi.std() * np.exp(1j * theta_test)

## get complex conjugate
varphi_test = np.stack([varphi_test, np.conj(varphi_test)], axis=0)

## Get recon (EOF space)
recon = np.einsum(
    "nmk,kn->mn",
    Uk[month_idx_test],
    varphi_test,
).real

## Get recon (real space)
recon_xr = reconstruct(recon, other_coord=pd.Index(lags_months, name="lag"))

In [ ]:
## get meridional means
lat = dict(latitude=slice(-5, 5))
recon_merimean = recon_xr.sel(lat).mean("latitude")
# comp_merimean = comp.sel(lat).mean("latitude")

## set up plot
fig, axs = plt.subplots(1, 2, figsize=(6, 3), layout="constrained")

for ax, merimean in zip(axs[:1], [recon_merimean]):

    ## SST
    ax.contourf(
        merimean.longitude,
        merimean.lag,
        merimean["T"],
        cmap="cmo.balance",
        levels=src.utils.make_cb_range(3, 0.3),
        extend="both",
    )

    ## thermocline
    ax.contour(
        merimean.longitude,
        merimean.lag,
        merimean["h"],
        colors="k",
        levels=src.utils.make_cb_range(9, 1.5),
        extend="both",
        linewidths=1,
    )

    ## label x axis
    ax.set_xticks([190, 240])
    ax.set_xlim([120, 280])

## label
axs[0].set_yticks([-12, 0, 12], labels=["Jan(-1)", "Jan(0)", "Jan(+1)"])
axs[1].set_yticks([])
# ax.set_xticks([190, 240])

plt.show()